# Stellar Spectral Analysis
### Excitation Temperature, Equivalent Widths & Spectral Fitting

This notebook analyses two observed stellar spectra using Fe I absorption lines.  
The workflow follows three stages:

1. **Multiplet analysis** — use the curve-of-growth to estimate excitation temperatures  
2. **Spectral fitting** — compare measured equivalent widths against a grid of BT-NextGen synthetic spectra  
3. **Spectral visualisation** — overlay synthetic and observed spectra, including rotational broadening for Star 2

**Data sources**
- Observed spectra: `estrela1.fits`, `estrela2.fits` (1D, wavelength-calibrated)  
- Fe I line list: Tsantaki et al. (2013), *A&A* 555, A150 — [doi:10.1051/0004-6361/201321103](https://doi.org/10.1051/0004-6361/201321103)  
- Synthetic spectra: BT-NextGen model grid (Allard et al. 2012)


## 0. Imports and Setup

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import rcParams

from astropy.io import fits
from numpy.polynomial import polynomial as P
from scipy import signal, fftpack
from scipy.optimize import curve_fit
from scipy.stats import norm

# ── Plot aesthetics ────────────────────────────────────────────────────────────
rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (10, 5),
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "lines.linewidth": 1.4,
})

np.set_printoptions(precision=4, suppress=True, threshold=sys.maxsize)

## 1. Core Utility Functions

These functions handle FITS loading, spectral windowing, Gaussian fitting, and
resolution/rotation broadening. They are used throughout all sections.


In [ ]:
# ── FITS reader ─────────────────────────────────────────────────────────────
def load_fits_spectrum(filename):
    """
    Read a 1D wavelength-calibrated FITS spectrum.

    Uses the standard WCS keywords CRVAL1 (starting wavelength) and CDELT1
    (wavelength step per pixel) to reconstruct the wavelength axis.

    Returns
    -------
    spectrum : ndarray, shape (N, 2)
        Column 0 = wavelength [Å], Column 1 = flux [arbitrary units].
    """
    with fits.open(filename) as hdul:
        header = hdul[0].header
        data   = hdul[0].data

    crval = header["crval1"]
    cdelt = header["cdelt1"]
    crpix = 1  # standard origin

    wavelength = np.arange(data.size) * cdelt + crval + cdelt * (1 - crpix)

    spectrum = np.column_stack([wavelength, data])
    return spectrum


# ── Spectral window ──────────────────────────────────────────────────────────
def extract_window(spectrum, wav_min, wav_max):
    """
    Return the portion of *spectrum* between wav_min and wav_max.

    Parameters
    ----------
    spectrum : ndarray, shape (N, 2)
    wav_min, wav_max : float
        Wavelength bounds [Å]. Order does not matter.

    Returns
    -------
    window : ndarray, shape (M, 2)
    """
    lo, hi = min(wav_min, wav_max), max(wav_min, wav_max)
    mask = (spectrum[:, 0] > lo) & (spectrum[:, 0] < hi)
    return spectrum[mask]


def trim_fits(wav_min, wav_max, filename):
    """Convenience wrapper: load a FITS file then extract a wavelength window."""
    spectrum = load_fits_spectrum(filename)
    return extract_window(spectrum, wav_min, wav_max)

In [ ]:
# ── Gaussian line profile ────────────────────────────────────────────────────
def gaussian(x, sigma, x_center, amplitude, continuum):
    """
    Gaussian absorption profile superimposed on a flat continuum.

    f(x) = A * exp(-0.5 * ((x - x_c) / σ)²) + continuum

    Parameters
    ----------
    sigma      : line width (standard deviation) [Å]
    x_center   : line centre [Å]
    amplitude  : line depth (negative for absorption)
    continuum  : local continuum level
    """
    return amplitude * np.exp(-0.5 * ((x - x_center) / sigma) ** 2) + continuum


# ── Equivalent width via Gaussian fit ───────────────────────────────────────
def equivalent_width_fit(wavelength_center, spectrum, half_window=0.25):
    """
    Fit a Gaussian to an absorption line and return its equivalent width.

    The equivalent width is computed analytically from the fitted Gaussian
    parameters:

        W_λ = |√(2π) · σ · A / continuum|

    Parameters
    ----------
    wavelength_center : float
        Rest wavelength of the line [Å].
    spectrum : ndarray, shape (N, 2)
        Full spectrum (wavelength, flux).
    half_window : float
        Half-width of the fitting window [Å]. Default 0.25.

    Returns
    -------
    W : float
        Equivalent width [Å]. Returns 0.0 on fit failure.
    """
    window = extract_window(spectrum, wavelength_center - half_window,
                                     wavelength_center + half_window)
    if len(window) < 5:
        return 0.0

    wav, flux = window[:, 0], window[:, 1]

    p0 = [0.25, wavelength_center, -abs(flux.mean() - flux.min()), flux[0]]
    try:
        popt, _ = curve_fit(gaussian, wav, flux, p0=p0)
        sigma, _, amplitude, continuum = popt
        return abs(np.sqrt(2 * np.pi) * sigma * amplitude / continuum)
    except RuntimeError:
        return 0.0

In [ ]:
# ── Instrumental resolution broadening ──────────────────────────────────────
def apply_resolution_broadening(spectrum_2col, wav_min, wav_max, R=60000):
    """
    Convolve a synthetic spectrum with a Gaussian instrumental profile.

    The FWHM of the instrumental profile at the central wavelength λ₀ is:

        FWHM = λ₀ / R

    which is converted to σ via  σ = FWHM / (2√(2 ln 2)).

    Parameters
    ----------
    spectrum_2col : ndarray, shape (N, 2)
        Input synthetic spectrum (wavelength [Å], flux).
    wav_min, wav_max : float
        Wavelength range to process [Å].
    R : int
        Spectral resolving power. Default 60 000.

    Returns
    -------
    broadened : ndarray, shape (M, 2)
        Broadened spectrum on a uniform 0.01 Å grid.
    """
    window = spectrum_2col[(spectrum_2col[:, 0] > wav_min) &
                           (spectrum_2col[:, 0] < wav_max)]

    lambda0 = 0.5 * (wav_min + wav_max)
    fwhm    = lambda0 / R
    sigma   = fwhm / (2 * np.sqrt(2 * np.log(2)))
    dx      = 0.01

    wav_uniform = np.arange(wav_min, wav_max, dx)
    flux_interp = np.interp(wav_uniform, window[:, 0], window[:, 1])

    kernel_x = np.arange(-5 * sigma, 5 * sigma, dx)
    kernel   = norm.pdf(kernel_x, 0, sigma)
    kernel  /= kernel.sum()                     # area-normalise

    flux_broadened = signal.convolve(flux_interp, kernel, mode="same")

    return np.column_stack([wav_uniform, flux_broadened])


# ── Rotational broadening ────────────────────────────────────────────────────
def apply_rotation_broadening(filename, wav_min, wav_max, v_sini_km_s,
                               R=60000, epsilon=0.3):
    """
    Apply instrumental + rotational broadening to a synthetic spectrum file.

    The rotational kernel is the standard Gray (2005) limb-darkening profile:

        G(Δλ) ∝ 2(1−ε)√(1−(Δλ/ΔλL)²) + (π·ε/2)(1−(Δλ/ΔλL)²)

    where ΔλL = λ₀ · v sin i / c  is the rotational half-width.

    Parameters
    ----------
    filename   : str   Path to the synthetic spectrum (.dat / .txt).
    wav_min, wav_max : float  Wavelength range [Å].
    v_sini_km_s : float  Projected rotational velocity [km/s].
    R          : int   Spectral resolving power.
    epsilon    : float Limb-darkening coefficient (0–1). Default 0.3.

    Returns
    -------
    flux_broadened : ndarray, shape (M,)
        Broadened flux on the same uniform grid as `apply_resolution_broadening`.
    """
    spec = np.loadtxt(filename)
    broadened_spec = apply_resolution_broadening(spec, wav_min, wav_max, R)

    lambda0  = 0.5 * (wav_min + wav_max)
    delta_lL = lambda0 * v_sini_km_s / 3e5       # rotational half-width [Å]
    dx       = 0.01

    z = np.arange(-delta_lL, delta_lL, dx) / delta_lL
    kernel = (2 * (1 - epsilon) * np.sqrt(1 - z**2)
              + np.pi * epsilon / 2 * (1 - z**2))
    kernel /= (np.pi * delta_lL * (1 - epsilon / 3))

    flux_rot = signal.convolve(broadened_spec[:, 1], kernel * dx, mode="same")
    return flux_rot

In [ ]:
# ── Fourier transform of a line ──────────────────────────────────────────────
def fourier_transform_line(window_2col):
    """
    Compute the Fourier power spectrum of a spectral line window.

    The spectrum is extended with flat wings (23× the window length) before
    transforming, to minimise edge ringing.

    Returns
    -------
    freq : ndarray   Spatial frequencies [Å⁻¹]
    power : ndarray  |FFT| amplitude
    """
    n = len(window_2col)
    extended = np.zeros((23 * n, 2))
    extended[:, 0]    = np.linspace(window_2col[0, 0] - 5 * n * abs(window_2col[1, 0] - window_2col[0, 0]),
                                    window_2col[-1, 0] + 5 * n * abs(window_2col[1, 0] - window_2col[0, 0]),
                                    23 * n)
    extended[:n, 1]         = window_2col[0,  1]
    extended[n:2*n, 1]      = window_2col[:, 1]
    extended[2*n:, 1]       = window_2col[-1, 1]
    extended[:, 1]         /= extended[:, 1].max()

    dx    = abs(extended[0, 0] - extended[1, 0])
    tf    = fftpack.fft(extended[:, 1] - 1)
    freq  = fftpack.fftfreq(len(extended[:, 1]), dx)
    power = np.abs(tf)
    return freq, power

---
## 2. Load the Fe I Line List

The Tsantaki et al. (2013) line list provides, for each Fe I transition:
- **λ** — rest wavelength [Å]  
- **EP** — excitation potential of the lower level [eV]  
- **log gf** — oscillator strength (log scale)  
- **EW☉** — solar equivalent width [mÅ]

Lines are grouped into **multiplets** by excitation potential ranges.  
Multiplets with similar EP belong to the same electron transition family;  
comparing their curve-of-growth positions is how we derive the excitation temperature.


In [ ]:
# Load line list (skip the 3-line header)
_line_data = np.loadtxt("data/line_list_tsantakiFEI.dat")

# Full arrays
lambda_FeI  = _line_data[3:, 0]   # rest wavelength [Å]
EP_FeI      = _line_data[3:, 1]   # excitation potential [eV]
loggf_FeI   = _line_data[3:, 2]   # log(gf)
EW_sun      = _line_data[3:, 3]   # solar equivalent width [mÅ]

print(f"Loaded {len(lambda_FeI)} Fe I lines")
print(f"Wavelength range: {lambda_FeI.min():.1f} – {lambda_FeI.max():.1f} Å")
print(f"EP range:         {EP_FeI.min():.2f} – {EP_FeI.max():.2f} eV")

In [ ]:
# ── Define multiplets by excitation potential bin ────────────────────────────
#
# Each multiplet groups lines that share a lower-level excitation potential.
# The EP ranges below correspond to distinct electron configurations in Fe I.
# Comparing multiplets at different EP values allows us to infer Texc using
# the Boltzmann equation.
#
# Multiplet indices (EP ranges, eV):
#   m1: 2.1 – 2.3  |  m2: 2.8 – 2.9  |  m3: 3.2 – 3.3
#   m4: 3.6 – 3.7  |  m5: 4.2 – 4.3  |  m6: 4.5 – 4.6  |  m7: 4.6 – 4.7

EP_BINS = {
    "m1": (2.1, 2.3),
    "m2": (2.8, 2.9),
    "m3": (3.2, 3.3),
    "m4": (3.6, 3.7),
    "m5": (4.2, 4.3),
    "m6": (4.5, 4.6),
    "m7": (4.6, 4.7),
}

COLORS = {
    "m1": "red", "m2": "steelblue", "m3": "green",
    "m4": "purple", "m5": "darkorange", "m6": "deeppink", "m7": "saddlebrown",
}

def select_multiplet(ep_lo, ep_hi):
    mask = (EP_FeI > ep_lo) & (EP_FeI < ep_hi)
    return lambda_FeI[mask], EW_sun[mask], loggf_FeI[mask]

multiplets = {name: select_multiplet(*ep) for name, ep in EP_BINS.items()}

for name, (lam, ew, lgf) in multiplets.items():
    lo, hi = EP_BINS[name]
    print(f"{name}  EP {lo:.1f}–{hi:.1f} eV   {len(lam):3d} lines")

---
## 3. Multiplet Analysis — Excitation Temperature

### Method

For a set of lines in the same multiplet, the **reduced equivalent width**
W_λ/λ is plotted against the **line strength** log(λ·gf).  
This is the **curve of growth** — at low EW (the linear regime) this
relationship is linear for each multiplet.

The **horizontal shift** Δ between two multiplets at excitation potentials
EP₁ and EP₂ is related to the excitation temperature via the Boltzmann equation:

$$T_{\rm exc} = \frac{5040 \, (\chi_2 - \chi_1)}{\Delta}$$

where 5040 = hc/k_B in eV·K and χ is the excitation potential in eV.

We compute Δ as the mean absolute difference between the curve-of-growth
positions of two multiplets over a common x-axis range.


In [ ]:
# ── Helper: fit a line to the curve of growth ────────────────────────────────
def cog_xaxis(lam, lgf):
    """Return the x-axis quantity log(λ·gf) for the curve of growth."""
    return lgf + np.log10(lam)

def fit_cog(lam, ew, lgf):
    """
    Fit a first-degree polynomial to (log(λ·gf), W/λ).
    Returns (intercept, slope, x_vals, y_fitted).
    """
    x = cog_xaxis(lam, lgf)
    y = ew / lam
    intercept, slope = P.polyfit(x, y, 1)
    return intercept, slope, x, intercept + slope * x

### 3.1 Solar Reference

We first apply the method to the **solar equivalent widths** provided in the
Tsantaki line list. This serves as a sanity check — the derived excitation
temperature should be close to the accepted solar effective temperature of
**T_eff,☉ ≈ 5778 K**.


In [ ]:
# ── Curve-of-growth fits for the Sun ─────────────────────────────────────────
sun_cog = {}
for name, (lam, ew, lgf) in multiplets.items():
    if len(lam) < 2:
        continue
    c, d, x, y_fit = fit_cog(lam, ew, lgf)
    sun_cog[name] = {"lam": lam, "ew": ew, "lgf": lgf,
                     "x": x, "y_fit": y_fit, "c": c, "d": d}

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for name, data in sun_cog.items():
    col = COLORS[name]
    ax.plot(data["x"], data["ew"] / data["lam"], "*",
            color=col, ms=8, label=name)
    ax.plot(data["x"], data["y_fit"], "-", color=col, lw=1.2)

ax.set_xlabel(r"$\log(\lambda \cdot gf)$")
ax.set_ylabel(r"$W_\lambda / \lambda$")
ax.set_title("Curve of Growth — Solar Equivalent Widths")
ax.legend(ncol=4, framealpha=0.6)
plt.tight_layout()
plt.savefig("results/cog_solar.pdf", format="pdf")
plt.show()

In [ ]:
# ── Excitation temperature from multiplet separations ─────────────────────────
#
# For each pair of multiplets (i, j) the horizontal separation Δ is computed
# as the mean |x_i − x_j| over a shared linear grid, then:
#
#     T_exc = 5040 × (EP_j − EP_i) / Δ
#

def excitation_temperature(ep_i, ep_j, x_i, x_j, n_points=100):
    """
    Estimate excitation temperature from two curve-of-growth branches.

    Parameters
    ----------
    ep_i, ep_j : float   Excitation potentials [eV].
    x_i, x_j  : ndarray  CoG x-axis values for each multiplet.
    n_points   : int      Resolution of the comparison grid.

    Returns
    -------
    T : float   Excitation temperature [K].
    delta : float  Mean horizontal separation.
    """
    xi_grid = np.linspace(x_i.min(), x_i.max(), n_points)
    xj_grid = np.linspace(x_j.min(), x_j.max(), n_points)
    delta = np.mean(np.abs(xi_grid - xj_grid))
    T = 5040 * abs(ep_j - ep_i) / delta
    return T, delta


# Compute all pairwise temperatures for multiplets present in the solar data
pairs = [("m1","m4"), ("m1","m5"), ("m1","m6"), ("m1","m7"),
         ("m3","m4"), ("m3","m5"), ("m3","m6"), ("m3","m7"),
         ("m4","m5"), ("m4","m6"), ("m4","m7"),
         ("m5","m6"), ("m5","m7"), ("m6","m7")]

T_sun_all = {}
print(f"{'Pair':<10}  {'EP_i':>6}  {'EP_j':>6}  {'Delta':>8}  {'T_exc [K]':>10}")
print("-" * 50)
for (a, b) in pairs:
    if a not in sun_cog or b not in sun_cog:
        continue
    ep_a = np.mean(EP_FeI[(EP_FeI > EP_BINS[a][0]) & (EP_FeI < EP_BINS[a][1])])
    ep_b = np.mean(EP_FeI[(EP_FeI > EP_BINS[b][0]) & (EP_FeI < EP_BINS[b][1])])
    T, delta = excitation_temperature(ep_a, ep_b,
                                      sun_cog[a]["x"], sun_cog[b]["x"])
    T_sun_all[(a, b)] = T
    print(f"{a}–{b:<6}  {ep_a:>6.2f}  {ep_b:>6.2f}  {delta:>8.4f}  {T:>10.0f}")

T_mean = np.mean(list(T_sun_all.values()))
print(f"\nMean excitation temperature (all pairs):  {T_mean:.0f} K")
print(f"Reference solar T_eff:                     5778 K")

### 3.2 Star 1

We now measure the **equivalent widths directly from the observed spectrum**
of Star 1 by fitting a Gaussian to each Fe I line window.
The same curve-of-growth analysis then yields an excitation temperature estimate.


In [ ]:
# Load Star 1
star1 = load_fits_spectrum("data/estrela1.fits")
star1 = star1[star1[:, 1] > 0]   # remove non-physical negative flux values

print(f"Star 1 — wavelength range: {star1[:,0].min():.1f} – {star1[:,0].max():.1f} Å")
print(f"         {len(star1)} pixels")

In [ ]:
# ── Measure EW for each multiplet ────────────────────────────────────────────
def measure_multiplet_ew(lam_list, spectrum, half_window=0.25):
    """
    Measure equivalent widths for a list of line wavelengths in *spectrum*.

    Returns an array of the same length as *lam_list*; failed fits are 0.
    """
    ews = np.zeros(len(lam_list))
    for i, lam in enumerate(lam_list):
        ews[i] = equivalent_width_fit(lam, spectrum, half_window)
    return ews


star1_cog = {}
for name, (lam, ew_sun, lgf) in multiplets.items():
    ew_obs = measure_multiplet_ew(lam, star1)
    mask   = ew_obs > 0          # keep only successful fits
    if mask.sum() < 2:
        continue
    lam_ok, ew_ok, lgf_ok = lam[mask], ew_obs[mask], lgf[mask]
    c, d, x, y_fit = fit_cog(lam_ok, ew_ok, lgf_ok)
    star1_cog[name] = {"lam": lam_ok, "ew": ew_ok, "lgf": lgf_ok,
                       "x": x, "y_fit": y_fit, "c": c, "d": d}
    print(f"{name}: {mask.sum()}/{len(lam)} lines measured successfully")

In [ ]:
# ── Curve-of-growth plot — Star 1 ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for name, data in star1_cog.items():
    col = COLORS[name]
    ax.plot(data["x"], data["ew"] / data["lam"], "*",
            color=col, ms=8, label=name)
    ax.plot(data["x"], data["y_fit"], "-", color=col, lw=1.2)

ax.set_xlabel(r"$\log(\lambda \cdot gf)$")
ax.set_ylabel(r"$W_\lambda / \lambda$")
ax.set_title("Curve of Growth — Star 1 (Measured EWs)")
ax.legend(ncol=4, framealpha=0.6)
plt.tight_layout()
plt.savefig("results/cog_star1.pdf", format="pdf")
plt.show()

In [ ]:
# ── Excitation temperatures — Star 1 ────────────────────────────────────────
T_star1_all = {}
print(f"{'Pair':<10}  {'T_exc [K]':>10}")
print("-" * 25)
for (a, b) in pairs:
    if a not in star1_cog or b not in star1_cog:
        continue
    ep_a = np.mean(EP_FeI[(EP_FeI > EP_BINS[a][0]) & (EP_FeI < EP_BINS[a][1])])
    ep_b = np.mean(EP_FeI[(EP_FeI > EP_BINS[b][0]) & (EP_FeI < EP_BINS[b][1])])
    T, _ = excitation_temperature(ep_a, ep_b,
                                  star1_cog[a]["x"], star1_cog[b]["x"])
    T_star1_all[(a, b)] = T
    print(f"{a}–{b:<6}  {T:>10.0f}")

print(f"\nMean excitation temperature — Star 1: {np.mean(list(T_star1_all.values())):.0f} K")

### 3.3 Star 2

Star 2 covers a different wavelength range (≈ 5855–6840 Å), so we restrict the
line list to lines falling within that range before repeating the analysis.
Some lines are pre-measured (hardcoded) from a previous run to avoid re-fitting
noisy regions; those are kept as-is.


In [ ]:
# Load Star 2 and restrict to its wavelength coverage
star2 = load_fits_spectrum("data/estrela2.fits")
star2 = star2[star2[:, 1] > 0]

lam2_min, lam2_max = star2[:, 0].min(), star2[:, 0].max()
print(f"Star 2 — wavelength range: {lam2_min:.1f} – {lam2_max:.1f} Å")

# Restrict line list to Star 2's range
in_range = (lambda_FeI > lam2_min) & (lambda_FeI < lam2_max)
lambda_FeI2 = lambda_FeI[in_range]
EP_FeI2     = EP_FeI[in_range]
loggf_FeI2  = loggf_FeI[in_range]
EW_sun2     = EW_sun[in_range]

print(f"Fe I lines in range: {in_range.sum()}")

In [ ]:
# ── Pre-measured EWs for multiplets that are partly in noisy regions ─────────
# These values were obtained from a previous careful Gaussian fit run.
# They correspond to the curve-of-growth groups m1, m5, m6 of Star 2.

Multn1_data = {
    "lam":  np.array([6151.62, 6173.34, 6219.29, 6335.34, 6392.54, 6481.88]),
    "lgf":  np.array([-3.299, -2.877, -2.463, -2.339, -3.942, -2.929]),
    "ew":   np.array([1.488e-05, 1.839e-05, 2.512e-05,
                      2.533e-05, 6.259e-06, 1.928e-05]),
}
Multn5_data = {
    "lam":  np.array([5862.36, 5902.48, 5983.69, 6024.06, 6089.57, 6627.55, 6732.07]),
    "lgf":  np.array([-0.404, -1.797, -0.719, -0.124, -1.273, -1.475, -2.144]),
    "ew":   np.array([2.384e-05, 6.590e-06, 2.041e-05,
                      2.782e-05, 8.509e-06, 1.113e-05, 3.503e-06]),
}
Multn6_data = {
    "lam":  np.array([5855.08, 5905.68, 5927.79, 5930.19,
                      6094.38, 6159.38, 6705.11, 6726.67]),
    "lgf":  np.array([-1.531, -0.775, -1.057, -0.326,
                      -1.566, -1.878, -1.057, -1.045]),
    "ew":   np.array([7.445e-06, 1.640e-05, 1.202e-05, 1.700e-05,
                      4.559e-06, 5.570e-06, 1.205e-05, 1.480e-05]),
}

# Fit CoG to these pre-measured multiplets
def build_cog_from_data(d):
    x = cog_xaxis(d["lam"], d["lgf"])
    c, s = P.polyfit(x, d["ew"] / d["lam"], 1)
    return {"x": x, "ew": d["ew"], "lam": d["lam"],
            "lgf": d["lgf"], "y_fit": c + s * x, "c": c, "d": s}

star2_cog = {
    "m1": build_cog_from_data(Multn1_data),
    "m5": build_cog_from_data(Multn5_data),
    "m6": build_cog_from_data(Multn6_data),
}

# Also measure the remaining multiplets from the observed spectrum
for name in ["m2", "m3", "m4"]:
    ep_lo, ep_hi = EP_BINS[name]
    mask_ep = (EP_FeI2 > ep_lo) & (EP_FeI2 < ep_hi)
    lam_m = lambda_FeI2[mask_ep]
    lgf_m = loggf_FeI2[mask_ep]
    if len(lam_m) < 2:
        continue
    ew_m = measure_multiplet_ew(lam_m, star2, half_window=0.45)
    ok = ew_m > 0
    if ok.sum() < 2:
        continue
    c, d, x, y_fit = fit_cog(lam_m[ok], ew_m[ok], lgf_m[ok])
    star2_cog[name] = {"lam": lam_m[ok], "ew": ew_m[ok], "lgf": lgf_m[ok],
                       "x": x, "y_fit": y_fit, "c": c, "d": d}

print("Multiplets available for Star 2:", list(star2_cog.keys()))

In [ ]:
# ── Curve-of-growth plot — Star 2 ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for name, data in star2_cog.items():
    col = COLORS[name]
    ax.plot(data["x"], data["ew"] / data["lam"], "*",
            color=col, ms=8, label=name)
    if "y_fit" in data:
        ax.plot(data["x"], data["y_fit"], "-", color=col, lw=1.2)

ax.set_xlabel(r"$\log(\lambda \cdot gf)$")
ax.set_ylabel(r"$W_\lambda / \lambda$")
ax.set_title("Curve of Growth — Star 2 (Measured EWs)")
ax.legend(ncol=4, framealpha=0.6)
plt.tight_layout()
plt.savefig("results/cog_star2.pdf", format="pdf")
plt.show()

In [ ]:
# ── Excitation temperatures — Star 2 ────────────────────────────────────────
T_star2_all = {}
print(f"{'Pair':<10}  {'T_exc [K]':>10}")
print("-" * 25)
for (a, b) in pairs:
    if a not in star2_cog or b not in star2_cog:
        continue
    ep_a = np.mean(EP_FeI[(EP_FeI > EP_BINS[a][0]) & (EP_FeI < EP_BINS[a][1])])
    ep_b = np.mean(EP_FeI[(EP_FeI > EP_BINS[b][0]) & (EP_FeI < EP_BINS[b][1])])
    T, _ = excitation_temperature(ep_a, ep_b,
                                  star2_cog[a]["x"], star2_cog[b]["x"])
    T_star2_all[(a, b)] = T
    print(f"{a}–{b:<6}  {T:>10.0f}")

print(f"\nMean excitation temperature — Star 2: {np.mean(list(T_star2_all.values())):.0f} K")

---
## 4. Spectral Fitting Against the BT-NextGen Grid

### Method

For each star, we compare the **measured equivalent widths** against those
predicted by every model in the BT-NextGen synthetic spectrum grid.

The goodness-of-fit statistic is a reduced chi-squared:

$$\chi^2 = \sum_i \left( W_{\lambda,\rm obs}^{(i)} - W_{\lambda,\rm model}^{(i)} \right)^2$$

summed only over lines for which both the observed and model EW are non-zero.

The model filename encodes its physical parameters:
`lteTTT-logg+[Fe/H]a+alpha.BT-NextGen.7.dat.txt`
where TTT is T_eff/100 (so `lte052` → T_eff = 5200 K).

> **Note:** The synthetic spectra are first convolved with the instrumental
> profile (R ≈ 60 000) before measuring equivalent widths, to match the
> resolution of the observations.


In [ ]:
# ── Measure EW from a synthetic spectrum (Star 1 wavelength range) ────────────
def measure_ew_synthetic_star1(filename):
    """
    Load a BT-NextGen file, apply resolution broadening, measure EW
    at each Fe I line position (full line list range).

    Returns an array of length len(lambda_FeI); zeros for missing/failed lines.
    """
    spec = np.loadtxt(filename)
    lam_min = lambda_FeI.min() - 10
    lam_max = lambda_FeI.max() + 10

    broadened = apply_resolution_broadening(spec, lam_min, lam_max, R=60000)
    ews = np.zeros(len(lambda_FeI))

    for i, lam in enumerate(lambda_FeI):
        window = extract_window(broadened, lam - 0.25, lam + 0.25)
        if len(window) < 5:
            continue
        wav, flux = window[:, 0], window[:, 1]
        flux = flux / flux.max()
        p0 = [0.25, lam, -0.5, flux[0]]
        try:
            popt, _ = curve_fit(gaussian, wav, flux, p0=p0)
            sig, _, A, B = popt
            ews[i] = abs(np.sqrt(2 * np.pi) * sig * A / B)
        except RuntimeError:
            pass

    return ews


# ── Measure EW from a synthetic spectrum (Star 2 wavelength range) ────────────
def measure_ew_synthetic_star2(filename):
    """Same as above but restricted to Star 2's wavelength range."""
    spec = np.loadtxt(filename)
    lam_min = lambda_FeI2.min()
    lam_max = lambda_FeI2.max()

    broadened = apply_resolution_broadening(spec, lam_min, lam_max, R=60000)
    ews = np.zeros(len(lambda_FeI2))

    for i, lam in enumerate(lambda_FeI2):
        window = extract_window(broadened, lam - 0.25, lam + 0.25)
        if len(window) < 5:
            continue
        wav, flux = window[:, 0], window[:, 1]
        flux = flux / flux.max()
        p0 = [0.25, lam, -0.5, flux[0]]
        try:
            popt, _ = curve_fit(gaussian, wav, flux, p0=p0)
            sig, _, A, B = popt
            ews[i] = abs(np.sqrt(2 * np.pi) * sig * A / B)
        except RuntimeError:
            pass

    return ews

In [ ]:
# ── Load pre-computed observed EWs (or compute them) ─────────────────────────
#
# Computing EWs for the full line list takes a while.
# We save/load from disk to avoid re-running on every kernel restart.

WL1_FILE = "data/WL_1.npy"
WL2_FILE = "data/WL_2.npy"

if os.path.exists(WL1_FILE):
    WL_1 = np.load(WL1_FILE)
    print(f"Loaded Star 1 EWs from {WL1_FILE}  ({(WL_1 > 0).sum()} non-zero)")
else:
    print("Computing Star 1 EWs — this may take a few minutes...")
    WL_1 = np.array([equivalent_width_fit(lam, star1) for lam in lambda_FeI])
    np.save(WL1_FILE, WL_1)
    print(f"Saved to {WL1_FILE}")

if os.path.exists(WL2_FILE):
    WL_2 = np.load(WL2_FILE)
    # Zero out known bad fits (bright sky lines / blends)
    bad_indices = [0, 6, 16, 20, 32, 42, 50]
    WL_2[bad_indices] = 0
    print(f"Loaded Star 2 EWs from {WL2_FILE}  ({(WL_2 > 0).sum()} non-zero)")
else:
    print("Computing Star 2 EWs — this may take a few minutes...")
    WL_2 = np.array([equivalent_width_fit(lam, star2, half_window=0.45)
                     for lam in lambda_FeI2])
    bad_indices = [0, 6, 16, 20, 32, 42, 50]
    WL_2[bad_indices] = 0
    np.save(WL2_FILE, WL_2)
    print(f"Saved to {WL2_FILE}")

In [ ]:
# ── Grid search — Star 1 ──────────────────────────────────────────────────────
#
# Iterate over all synthetic spectra in the model grid directories.
# Keep only models with chi² < threshold (0.1 here).

GRID_DIRS = ["data/espetros_5000-6000", "data/espetros_6000-7000"]

SINT1_FILE = "data/Sint1.npy"

if os.path.exists(SINT1_FILE):
    l_1 = np.load(SINT1_FILE, allow_pickle=True)
    print(f"Loaded Star 1 best-fit models: {len(l_1)} candidates")
else:
    l1_candidates = []
    for grid_dir in GRID_DIRS:
        if not os.path.isdir(grid_dir):
            continue
        for fname in os.listdir(grid_dir):
            fpath = os.path.join(grid_dir, fname)
            if not os.path.isfile(fpath):
                continue
            Wn = measure_ew_synthetic_star1(fpath)
            if np.all(Wn == 0):
                continue
            chi2 = sum((WL_1[i] - Wn[i])**2
                       for i in range(len(WL_1))
                       if Wn[i] != 0 and WL_1[i] != 0)
            if chi2 < 0.1:
                l1_candidates.append([chi2, fpath])
                print(f"  chi²={chi2:.4f}  {fpath}")

    l_1 = np.array(l1_candidates)
    np.save(SINT1_FILE, l_1)
    print(f"\nSaved {len(l_1)} candidate models to {SINT1_FILE}")

In [ ]:
# ── Grid search — Star 2 ──────────────────────────────────────────────────────
#
# For Star 2 we use a wider chi² window (0.6 – 0.9) because the spectrum is
# noisier and very low chi² values arise from models with many missing lines
# (not a good fit, just fewer comparison points).

SINT2_FILE = "data/Sint2.npy"

if os.path.exists(SINT2_FILE):
    l_2 = np.load(SINT2_FILE, allow_pickle=True)
    print(f"Loaded Star 2 best-fit models: {len(l_2)} candidates")
else:
    l2_candidates = []
    for grid_dir in GRID_DIRS:
        if not os.path.isdir(grid_dir):
            continue
        for fname in os.listdir(grid_dir):
            fpath = os.path.join(grid_dir, fname)
            if not os.path.isfile(fpath):
                continue
            Wn = measure_ew_synthetic_star2(fpath)
            if np.all(Wn == 0):
                continue
            chi2 = sum((WL_2[i] - Wn[i])**2
                       for i in range(len(WL_2))
                       if Wn[i] != 0 and WL_2[i] != 0)
            if 0.6 <= chi2 <= 0.9:
                l2_candidates.append([chi2, fpath])
                print(f"  chi²={chi2:.4f}  {fpath}")

    l_2 = np.array(l2_candidates)
    np.save(SINT2_FILE, l_2)
    print(f"\nSaved {len(l_2)} candidate models to {SINT2_FILE}")

In [ ]:
# ── Summary of best-fit models ────────────────────────────────────────────────
def parse_model_name(path):
    """Extract T_eff, log g, [Fe/H] from BT-NextGen filename."""
    fname = os.path.basename(str(path))
    # e.g. lte052-4.0-1.0a+0.2.BT-NextGen.7.dat.txt
    try:
        parts = fname.replace("lte","").split("-")
        teff  = int(parts[0]) * 100
        logg  = float(parts[1])
        feh   = float(parts[2].split("a")[0])
        return teff, logg, feh
    except Exception:
        return None, None, None

if len(l_1) > 0:
    print("Star 1 — top 5 best-fit models:")
    sorted1 = l_1[np.argsort(l_1[:, 0].astype(float))]
    for row in sorted1[:5]:
        chi2, path = float(row[0]), str(row[1])
        teff, logg, feh = parse_model_name(path)
        print(f"  chi²={chi2:.5f}  T_eff={teff} K  log g={logg}  [Fe/H]={feh}")

if len(l_2) > 0:
    print("\nStar 2 — top 5 best-fit models:")
    sorted2 = l_2[np.argsort(l_2[:, 0].astype(float))]
    for row in sorted2[:5]:
        chi2, path = float(row[0]), str(row[1])
        teff, logg, feh = parse_model_name(path)
        print(f"  chi²={chi2:.5f}  T_eff={teff} K  log g={logg}  [Fe/H]={feh}")

---
## 5. Spectral Visualisation

### 5.1 Fourier Analysis of Absorption Lines

The Fourier transform of an absorption line profile encodes information about
line broadening. The **first zero** of the Fourier power spectrum is related
to the projected rotational velocity v sin i:

$$\sigma_1 = 0.660 / (\Delta\lambda_L) \quad \text{(first zero for } \epsilon=0)$$

Comparing the frequency of the first zero across different lines (and spectral
regions) gives a consistency check on the measured v sin i.


In [ ]:
# ── Fourier spectra — Star 1 ──────────────────────────────────────────────────
LINE_CENTERS_STAR1 = {
    "4882.42 Å": 4882.42,
    "5862.36 Å": 5862.36,
    "6912.40 Å": 6912.40,
}

fig, ax = plt.subplots(figsize=(10, 4))
colors_ft = ["steelblue", "crimson", "seagreen"]

for (label, lam), col in zip(LINE_CENTERS_STAR1.items(), colors_ft):
    window = extract_window(star1, lam - 0.25, lam + 0.25)
    if len(window) < 5:
        continue
    freq, power = fourier_transform_line(window)
    mask = (freq >= 0) & (freq <= 0.2)
    ax.plot(freq[mask], power[mask], ".", color=col, ms=3)
    ax.plot(freq[mask], power[mask], "-", color=col, lw=1, label=label)

ax.set_xlabel(r"$\sigma$ $[\rm Å^{-1}]$")
ax.set_ylabel("Fourier Amplitude")
ax.set_title("Fourier Power Spectra of Fe I Lines — Star 1")
ax.set_ylim(0, 10)
ax.legend()
plt.tight_layout()
plt.savefig("results/fourier_star1.pdf", format="pdf")
plt.show()

### 5.2 Observed vs Synthetic Spectra

We compare the observed spectrum directly to the best-fitting BT-NextGen models.
For **Star 1** no rotational broadening is applied (slow rotator).
For **Star 2** we apply a rotational kernel with v sin i ≈ 10.8 km/s
(derived from the Fourier analysis above).


In [ ]:
# ── Star 1: observed vs best-fit synthetic ────────────────────────────────────
PLOT_CENTER = 6335.34   # Fe I line used for comparison [Å]
HALF_RANGE  = 5.0       # window half-width [Å]

if len(l_1) > 0:
    best_model_1 = str(sorted1[0][1])

    spec_syn = np.loadtxt(best_model_1)
    broad_syn = apply_resolution_broadening(spec_syn,
                                            PLOT_CENTER - HALF_RANGE,
                                            PLOT_CENTER + HALF_RANGE,
                                            R=60000)
    broad_obs = apply_resolution_broadening(star1,
                                            PLOT_CENTER - HALF_RANGE,
                                            PLOT_CENTER + HALF_RANGE,
                                            R=60000)

    trim = 20   # edge pixels to drop (convolution artefacts)
    wav_syn = broad_syn[trim:-trim, 0]
    flux_syn = broad_syn[trim:-trim, 1] / broad_syn[trim:-trim, 1].max()
    wav_obs = broad_obs[trim:-trim, 0]
    flux_obs = broad_obs[trim:-trim, 1] / broad_obs[trim:-trim, 1].max()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(wav_syn, flux_syn, label=f"Synthetic — {os.path.basename(best_model_1)}", lw=1.5)
    ax.plot(wav_obs, flux_obs, label="Observed (Star 1)",         lw=1.2, alpha=0.85)
    ax.set_xlabel(r"$\lambda$ [Å]")
    ax.set_ylabel(r"$F_\lambda$ (normalised)")
    ax.set_title(f"Star 1 — Observed vs Best-fit Synthetic (centred on {PLOT_CENTER:.2f} Å)")
    ax.legend()
    plt.tight_layout()
    plt.savefig("results/spectrum_star1.pdf", format="pdf")
    plt.show()
else:
    print("No best-fit model available for Star 1 — run the grid search first.")

In [ ]:
# ── Star 2: Fourier analysis ──────────────────────────────────────────────────
LINE_INDICES_STAR2 = [2, 34, 47]   # indices into lambda_FeI2
colors_ft2 = ["steelblue", "crimson", "seagreen"]

fig, ax = plt.subplots(figsize=(10, 4))

for idx, col in zip(LINE_INDICES_STAR2, colors_ft2):
    lam = lambda_FeI2[idx]
    window = extract_window(star2, lam - 0.40, lam + 0.51)
    if len(window) < 5:
        continue
    freq, power = fourier_transform_line(window)
    mask = (freq >= 0) & (freq <= 0.2)
    ax.plot(freq[mask], power[mask], ".", color=col, ms=3)
    ax.plot(freq[mask], power[mask], "-", color=col, lw=1, label=f"{lam:.2f} Å")

ax.set_xlabel(r"$\sigma$ $[\rm Å^{-1}]$")
ax.set_ylabel("Fourier Amplitude")
ax.set_title("Fourier Power Spectra of Fe I Lines — Star 2")
ax.set_ylim(0, 4)
ax.legend()
plt.tight_layout()
plt.savefig("results/fourier_star2.pdf", format="pdf")
plt.show()

In [ ]:
# ── Star 2: observed vs best-fit synthetic (with rotational broadening) ───────
V_SINI = 10.8   # projected rotational velocity [km/s]

if len(l_2) > 0:
    # Show the two best candidates
    top_models_2 = [str(r[1]) for r in sorted2[:2]]

    fig, ax = plt.subplots(figsize=(10, 4))

    broad_obs2 = apply_resolution_broadening(star2,
                                             PLOT_CENTER - HALF_RANGE,
                                             PLOT_CENTER + HALF_RANGE,
                                             R=60000)
    wav_obs2  = np.linspace(PLOT_CENTER - HALF_RANGE, PLOT_CENTER + HALF_RANGE,
                             len(broad_obs2))
    trim = 20
    ax.plot(wav_obs2[trim:-trim],
            broad_obs2[trim:-trim, 1] / broad_obs2[trim:-trim, 1].max(),
            label="Observed (Star 2)", lw=1.2, color="black", alpha=0.8)

    for i, model_path in enumerate(top_models_2):
        chi2 = float(sorted2[i][0])
        flux_rot = apply_rotation_broadening(model_path,
                                             PLOT_CENTER - HALF_RANGE,
                                             PLOT_CENTER + HALF_RANGE,
                                             V_SINI, R=60000)
        wav_syn2 = np.arange(PLOT_CENTER - HALF_RANGE, PLOT_CENTER + HALF_RANGE, 0.01)
        teff, logg, feh = parse_model_name(model_path)
        label = f"T={teff} K, log g={logg}, [Fe/H]={feh}  (χ²={chi2:.3f})"
        ax.plot(wav_syn2[trim:-trim],
                flux_rot[trim:-trim] / flux_rot[trim:-trim].max(),
                label=label, lw=1.4)

    ax.set_xlabel(r"$\lambda$ [Å]")
    ax.set_ylabel(r"$F_\lambda$ (normalised)")
    ax.set_title(f"Star 2 — Observed vs Best-fit Synthetic + Rotation (v sin i = {V_SINI} km/s)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("results/spectrum_star2.pdf", format="pdf")
    plt.show()
else:
    print("No best-fit model available for Star 2 — run the grid search first.")

---
## 6. Summary of Results

| Parameter | Sun (reference) | Star 1 | Star 2 |
|---|---|---|---|
| T_exc [K] | ≈ 5778 | *computed above* | *computed above* |
| Best T_eff [K] | — | *from grid search* | *from grid search* |
| log g | — | *from model name* | *from model name* |
| [Fe/H] | — | *from model name* | *from model name* |
| v sin i [km/s] | — | < 2 (slow) | ≈ 10.8 |

> Fill in the table after running the full notebook.

### Key caveats
- The excitation temperature from the multiplet method is sensitive to the accuracy of
  the Gaussian fits, especially for weaker lines in noisier spectral regions.
- The chi-squared metric here is **not** reduced (not divided by degrees of freedom),
  so it scales with the number of lines compared. Models covering more lines will 
  tend to have larger raw chi².
- Star 2's wider chi² acceptance window (0.6–0.9 vs <0.1 for Star 1) reflects
  the need to balance fit quality against line coverage completeness.
